# LemGendary Master Execution: ProfessionalMultitaskRestoration (v16.2 Nuclear-Hardened)
This unified notebook handles environment synchronization and automated cloud training.


## 1. Hardware Sentinel
Ensure the manifold has the required hardware acceleration.


In [ ]:
import torch, sys
print('[OK] [SENTINEL] Auditing Hardware Manifold...')
if not torch.cuda.is_available():
    print('[WARNING] NO GPU DETECTED!')
    print('[ACTION REQUIRED] Enable GPU Accelerator in notebook settings:')
    print('   -> Kaggle: Right Panel -> Session Options -> Accelerator -> GPU T4 x2 or P100')
    print('   -> Colab:  Runtime -> Change runtime type -> Hardware accelerator -> GPU')
    print('   -> Continuing in CPU Fallback Mode for dry-run validation...')
else:
    props = torch.cuda.get_device_properties(0)
    print(f'[OK] [ACTIVE] {props.name}')
    print(f'[OK] [VRAM] {props.total_memory / 1024**3:.1f} GB')
    if props.total_memory / 1024**3 < 10.0:
        print('[WARNING] Low VRAM detected. Suite will enable Survival Profiles automatically.')


## 2. Cloud Auth & Secrets


In [ ]:
try:
    from google.colab import userdata
    import os as _os, json as _json
    g_pat = None
    s_pat = None
    k_key = None
    k_user = None
    g_drive = None
    try: g_pat = userdata.get('GITHUB_PAT')
    except Exception as e: print(f'[REMEDY] Caught exception in notebook logic: {e}')
    try: s_pat = userdata.get('SUITE_PAT')
    except Exception as e: print(f'[REMEDY] Caught exception in notebook logic: {e}')
    try: k_key = userdata.get('KAGGLE_KEY')
    except Exception as e: print(f'[REMEDY] Caught exception in notebook logic: {e}')
    try: k_user = userdata.get('KAGGLE_USERNAME')
    except Exception as e: print(f'[REMEDY] Caught exception in notebook logic: {e}')
    try: g_drive = userdata.get('GOOGLE_DRIVE')
    except Exception as e: print(f'[REMEDY] Caught exception in notebook logic: {e}')
    
    if g_pat: _os.environ['GITHUB_PAT'] = g_pat
    if s_pat: _os.environ['SUITE_PAT'] = s_pat
    if g_drive: _os.environ['GOOGLE_DRIVE'] = g_drive
    
    if not k_user: k_user = 'lemtreursi'
    if k_key:
        _os.environ['KAGGLE_KEY'] = k_key
        _os.environ['KAGGLE_USERNAME'] = k_user
        _k_dir = _os.path.expanduser('~/.kaggle')
        _os.makedirs(_k_dir, exist_ok=True)
        with open(_os.path.join(_k_dir, 'kaggle.json'), 'w') as _kf:
            _json.dump({'username': k_user, 'key': k_key}, _kf)
        _os.chmod(_os.path.join(_k_dir, 'kaggle.json'), 0o600)
    
    active = []
    if s_pat: active.append('SUITE_PAT')
    if g_pat: active.append('GITHUB_PAT')
    if k_key: active.append('KAGGLE_KEY')
    if g_drive: active.append('GOOGLE_DRIVE')
    if active:
        print(f'[OK] [AUTH] Colab Secrets mounted: {", ".join(active)}')
    else:
        print('[WARNING] No PATs found in Colab Secrets! Private repositories will fail to clone.')
        print('[ACTION REQUIRED] Add SUITE_PAT or GITHUB_PAT to Colab Secrets.')
except Exception as e:
    print(f'[ERROR] Secret mounting failed: {e}')


## 3. Environment Synchronization


In [ ]:
import os, subprocess, shutil
repo_url = 'https://github.com/lemgenda/lemgendary-training-suite.git'
suite_path = '/content/lemgendary-training-suite'
pat = os.environ.get('SUITE_PAT', os.environ.get('GITHUB_PAT', ''))
if pat:
    # Use x-access-token for more reliable auth with fine-grained tokens
    auth_url = repo_url.replace('https://', f'https://x-access-token:{pat}@')
    print(f'[AUTH] Using {"SUITE_PAT" if os.environ.get("SUITE_PAT") else "GITHUB_PAT"} for cloning...')
else:
    print('[WARNING] No PAT found in environment. Attempting public clone (will fail for private repos)...')
    print('[ACTION REQUIRED] If clone fails, add SUITE_PAT or GITHUB_PAT to Kaggle Add-ons -> Secrets.')
    auth_url = repo_url

env = os.environ.copy()
env['GIT_TERMINAL_PROMPT'] = '0'

if not os.path.exists(suite_path):
    print('[SUITE] Initializing LemGendary Training Suite...')
    res = subprocess.run(['git', 'clone', auth_url, suite_path], capture_output=True, text=True, env=env)
    if res.returncode == 0: 
        print('[OK] Suite cloned.')
    else: 
        print(f'[ERROR] Clone failed: {res.stderr.strip()}')
        if '403' in res.stderr or '401' in res.stderr or 'terminal prompts disabled' in res.stderr:
            print('[ACTION REQUIRED] Add SUITE_PAT or GITHUB_PAT to Kaggle Add-ons -> Secrets with GitHub read permissions.')
else:
    print('[OK] Suite resident. Syncing origin and pulling latest...')
    subprocess.run(['git', 'remote', 'set-url', 'origin', auth_url], cwd=suite_path, env=env)
    subprocess.run(['git', 'pull'], cwd=suite_path, env=env)


In [ ]:
import os, sys, subprocess
print('[ENV] Installing Nuclear Dependencies...')
suite_candidates = ['/content/lemgendary-training-suite', '/content/model-training/lemgendary-training-suite', '/content']
req_path = next((os.path.join(p, 'requirements.txt') for p in suite_candidates if os.path.exists(os.path.join(p, 'requirements.txt'))), None)
if req_path:
    res = subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '--no-warn-conflicts', '--upgrade-strategy', 'only-if-needed', '-r', req_path])
    if res.returncode == 0:
        print('[OK] Environment Ready.')
    else:
        print('[WARNING] Dependency installation finished with non-zero exit code.')
else:
    print('[ERROR] Could not open requirements file: No such file or directory')
    print('[ACTION REQUIRED] Suite clone failed in Step 3 because SUITE_PAT/GITHUB_PAT is missing from Kaggle Secrets.')
    print('[ACTION REQUIRED] Fix: Go to Kaggle Notebook top bar -> Add-ons -> Secrets -> Add SUITE_PAT or GITHUB_PAT with your GitHub token.')


## 4. SOTA Hub Synchronization (Pull)


In [ ]:
import os
hub_root = '/content/LemGendaryModels'
model_key = 'professional_multitask_restoration'
model_dir = os.path.join(hub_root, model_key)
ckpt_dir = os.path.join(model_dir, 'checkpoints')

print(f'[HUB] Initializing Lean Manifold for {model_key}...')
os.makedirs(ckpt_dir, exist_ok=True)
print(f'[OK] Manifold structure ready at {model_dir}')


## 4.5 Google Drive Mount
Mount Google Drive FUSE for streaming datasets directly.


In [ ]:
import os
print('[MOUNT] Attaching Google Drive FUSE...')
from google.colab import drive
drive.mount('/content/drive')
print('[OK] Google Drive mounted successfully. Datasets will be streamed directly from Drive.')


## 5. Multi-Path Data Resolution


In [ ]:
import os
model_key = 'professional_multitask_restoration'
target_dir = '/content/LemGendaryDatasets'
os.makedirs(target_dir, exist_ok=True)

print(f'[DATA] Resolving manifolds for {model_key}...')
found = []
keys = [model_key.lower(), model_key.replace("_", "-"), model_key.replace("_", "")]

# 1. Restricted BFS Scanner (max depth 4, directories only) to bypass FUSE latency
if True:
    try:
        drives = [d for d in ['/content/drive/MyDrive', '/content/drive/Shareddrives', '/content/drive/Shared with me'] if os.path.exists(d)]
        if not drives: drives = ['/content/drive']
        queue = list(drives)
        depths = {d: 0 for d in drives}
        while queue:
            curr = queue.pop(0)
            depth = depths[curr]
            if depth > 4: continue
            for item in os.listdir(curr):
                path = os.path.join(curr, item)
                if os.path.isdir(path):
                    item_lower = item.lower()
                    # Prune models/checkpoints to prevent wasting time scanning weights
                    # 2026 Resilience: Aggressive FUSE Pruning - NEVER enter raw image/target dirs to prevent OOM stat storms
                    if item_lower in ['models', 'checkpoints', 'weights', 'images', 'targets', 'labels', 'masks', 'train', 'val', 'test', 'eval', 'lemgendarymodels']:
                        continue
                    depths[path] = depth + 1
                    queue.append(path)
                    
                    is_match = any(k in item_lower for k in keys) or 'lemgendary' in item_lower or 'datasets' in item_lower
                    if is_match:
                        def is_valid_ds(p):
                            try: return os.path.exists(os.path.join(p, 'images')) or os.path.exists(os.path.join(p, 'targets')) or 'forex' in os.path.basename(p).lower() or any(f.endswith('.csv') or f.endswith('.json') or f.endswith('.parquet') for f in os.listdir(p))
                            except: return False
                        if is_valid_ds(path): found.append(path)
                        else:
                            try:
                                for sub in os.listdir(path):
                                    sub_cand = os.path.join(path, sub)
                                    if os.path.isdir(sub_cand) and is_valid_ds(sub_cand): found.append(sub_cand)
                            except Exception as e: print(f'[REMEDY] Caught exception in notebook logic: {e}')
    except Exception:
        pass

for d in sorted(list(set(found))):
    if os.path.isdir(d):
        bname = os.path.basename(d)
        links = [bname]
        for link in links:
            link_name = os.path.join(target_dir, link)
            if not os.path.exists(link_name):
                try: os.symlink(d, link_name)
                except Exception as e: print(f'[REMEDY] Caught exception in notebook logic: {e}')
                print(f'[OK] [LINKED] {link} -> {d}')


## 6. Checkpoint & Metric Recovery


In [ ]:
import os, shutil
model_key = 'professional_multitask_restoration'
print(f'[RECOVERY] Deep-searching for {model_key} checkpoints...')
hub_root = '/content/LemGendaryModels'
model_hub_dir = os.path.join(hub_root, model_key)
ckpt_hub_dir = os.path.join(model_hub_dir, 'checkpoints')
os.makedirs(ckpt_hub_dir, exist_ok=True)

reg_filename = ''
try:
    import yaml
    yaml_path = '/content/lemgendary-training-suite/unified_models_v2.yaml'
    if os.path.exists(yaml_path):
        with open(yaml_path, 'r') as f: reg = yaml.safe_load(f)
        reg_filename = reg.get(model_key, {}).get('filename', '')
except Exception as e: print(f'[REMEDY] Caught exception in notebook logic: {e}')

target_slugs = [model_key.lower().replace('_', ''), model_key.lower().replace('_', '-'), reg_filename.lower() if reg_filename else '']
target_slugs = [s for s in target_slugs if s]

found_ckpts = []
if os.path.exists('/content/drive/MyDrive'):
    try:
        # Fast BFS Directory Search up to depth 7 to locate checkpoint folders
        queue = ['/content/drive/MyDrive']
        depths = {'/content/drive/MyDrive': 0}
        while queue:
            curr = queue.pop(0)
            depth = depths[curr]
            if depth > 7: continue
            for item in os.listdir(curr):
                path = os.path.join(curr, item)
                if os.path.isdir(path):
                    item_lower = item.lower()
                    # Prune image manifolds and datasets directory entirely to bypass FUSE latency
                    if item_lower in ['datasets', 'images', 'train', 'val', 'test', 'validation', 'dataset']:
                        continue
                    depths[path] = depth + 1
                    queue.append(path)
                    
                    # If matching candidate directory name, list the pth files
                    if any(slug in item_lower for slug in target_slugs) or 'checkpoint' in item_lower or 'weights' in item_lower or 'models' in item_lower:
                        try:
                            for f in os.listdir(path):
                                if f.lower().endswith('.pth') and (any(slug in f.lower() for slug in target_slugs) or 'best' in f.lower() or 'latest' in f.lower() or 'progress' in f.lower()):
                                    found_ckpts.append(os.path.join(path, f))
                        except:
                            pass
    except Exception:
        pass

found_ckpts = sorted(list(set(found_ckpts)))
if found_ckpts:
    print(f'   -> [FOUND] {len(found_ckpts)} binaries in Kaggle Manifold.')
    for src in found_ckpts:
        if f'/{model_key}/' not in src.replace('\\', '/') and f'{model_key}' not in os.path.basename(src):
            continue
        if not os.path.exists(src):
            print(f'   -> [WARNING] Source missing (Ghost File/Broken Link): {src}')
            continue
        fname = os.path.basename(src)
        target_f = fname
        if 'latest' in fname.lower(): target_f = f'{model_key}_latest.pth'
        elif 'best' in fname.lower(): target_f = f'{model_key}_best.pth'
        elif 'progress' in fname.lower(): target_f = f'{model_key}_progress.pth'
        
        dst = os.path.join(ckpt_hub_dir, target_f)
        if not os.path.exists(dst) or os.path.getsize(src) > os.path.getsize(dst):
            shutil.copy2(src, dst)
            print(f'   -> [OK] Recovered: {fname} -> {target_f}')
    
    metrics_found = False
    for src in found_ckpts:
        # Look for metrics.csv in parent or grandparent of the checkpoint
        for d in [os.path.dirname(os.path.dirname(src)), os.path.dirname(src)]:
            m_path = os.path.join(d, 'metrics.csv')
            if os.path.exists(m_path):
                try:
                    shutil.copy2(m_path, os.path.join(model_hub_dir, 'metrics.csv'))
                    print(f'[METRICS] Recovered metrics.csv from {os.path.basename(d)}')
                    metrics_found = True; break
                except Exception as e: print(f'[REMEDY] Caught exception in notebook logic: {e}')
        if metrics_found: break
else: print('   -> [SKIP] No existing checkpoints found in Kaggle Inputs manifold.')


## 7. Continuous Drive Synchronization


In [ ]:
import os, time, shutil, threading
model_key = 'professional_multitask_restoration'
hub_root = '/content/LemGendaryModels'
model_hub_dir = os.path.join(hub_root, model_key)
ckpt_hub_dir = os.path.join(model_hub_dir, 'checkpoints')

drive_target_dir = None
if found_ckpts:
    drive_target_dir = os.path.dirname(found_ckpts[0])

def drive_sync_worker():
    print(f'[SYNC] Background sync thread started. Target: {drive_target_dir}')
    while True:
        try:
            for f in os.listdir(ckpt_hub_dir):
                src = os.path.join(ckpt_hub_dir, f)
                if os.path.isfile(src):
                    dst = os.path.join(drive_target_dir, f)
                    # Copy if newer or doesn't exist
                    if not os.path.exists(dst) or os.path.getmtime(src) > os.path.getmtime(dst):
                        tmp_dst = dst + '.tmp'
                        shutil.copy2(src, tmp_dst)
                        os.rename(tmp_dst, dst)
            # Sync metrics.csv
            m_src = os.path.join(model_hub_dir, 'metrics.csv')
            if os.path.exists(m_src):
                m_dst = os.path.join(os.path.dirname(drive_target_dir), 'metrics.csv')
                if not os.path.exists(m_dst) or os.path.getmtime(m_src) > os.path.getmtime(m_dst):
                    shutil.copy2(m_src, m_dst)
        except Exception as e:
            pass
        time.sleep(30) # Sync every 30 seconds

if drive_target_dir:
    t = threading.Thread(target=drive_sync_worker, daemon=True)
    t.start()
else:
    print('[WARNING] No Google Drive checkpoint directory found. Background sync disabled.')


## 8. Nuclear Training Matrix


In [ ]:
import os, subprocess, sys
suite_candidates = ['/content/lemgendary-training-suite', '/content/model-training/lemgendary-training-suite', '/content']
active_suite_dir = next((p for p in suite_candidates if os.path.exists(os.path.join(p, 'training', 'train.py'))), '/content/lemgendary-training-suite')
os.chdir(active_suite_dir)
print(f'[OK] [SUITE] Active working directory set to: {os.getcwd()}')

# [JANITOR] Clean up any pre-existing zombie training processes to free the GPU
try:
    current_pid = os.getpid()
    ps_out = subprocess.check_output(['ps', '-ef'], text=True)
    for line in ps_out.split('\n'):
        if 'train.py' in line and str(current_pid) not in line:
            parts = line.split()
            if len(parts) > 1:
                pid = int(parts[1])
                print(f'[JANITOR] Killing stale zombie training process (PID {pid})...')
                subprocess.run(['kill', '-9', str(pid)], capture_output=True)
except Exception:
    pass

print(f'[LAUNCH] [NUCLEAR] Initiating Training Matrix for {model_key}...')
cmd = [sys.executable, '-u', 'training/train.py', '--model', f'{model_key}', '--env', 'colab', '--auto_sync']
p = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT)
try:
    import io
    for line in io.TextIOWrapper(p.stdout, newline=''):
        print(line, end='', flush=True)
    p.wait()
except KeyboardInterrupt:
    print('\n[TERMINATED] Training interrupted by user. Terminating training subprocess safely...')
    try:
        p.terminate()
        p.wait(timeout=5)
    except subprocess.TimeoutExpired:
        p.kill()
    print('[OK] Subprocess successfully killed. VRAM and CPU are clean.')
